# resNet18 with GA

In [ ]:
'''import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet18
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset
# ======================
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = DataLoader(testset, batch_size=64, shuffle=False)

# ======================
# ResNet-18 Feature Extractor
# ======================
model = resnet18(pretrained=True)
model.fc = nn.Identity()
model = model.to(device)
model.eval()

for p in model.parameters():
    p.requires_grad = False

# ======================
# Feature Extraction
# ======================
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats), torch.cat(labels)

print("Extracting features...")
X_train, y_train = extract_features(trainloader)
X_test,  y_test  = extract_features(testloader)

# ======================
# Train / Val Split (IMPORTANT)
# ======================
N = int(0.9 * len(X_train))
X_tr, X_val = X_train[:N], X_train[N:]
y_tr, y_val = y_train[:N], y_train[N:]

classifier = nn.Linear(512, 100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(classifier.parameters(), lr=0.01, momentum=0.9)

loader = DataLoader(
    TensorDataset(X_tr.to(device), y_tr.to(device)),
    batch_size=256, shuffle=True
)

print("\n=== SGD TRAINING ===")
for epoch in range(20):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")

# Save SGD weights
W_sgd = classifier.weight.data.T.cpu()   # (512 × 100)
b_sgd = classifier.bias.data.cpu()


D_HIGH = 1000

R = torch.randn(512, D_HIGH)
R, _ = torch.linalg.qr(R)   # 🔥 orthogonal

X_tr_h  = X_tr @ R
X_val_h = X_val @ R
X_te_h  = X_test @ R

W_high_init = R.T @ W_sgd   # (D_HIGH × 100)


# POP_SIZE = 20
# GENERATIONS = 20
# MUT_RATE = 0.3
# DELTA = 0.01

# def fitness(W):
#     with torch.no_grad():
#         logits = X_val_h @ W
#         loss = criterion(logits, y_val)
#     return -loss.item()

# population = [
#     W_high_init + 0.01 * torch.randn_like(W_high_init)
#     for _ in range(POP_SIZE)
# ]

# print("\n=== GA OPTIMIZATION ===")
# for gen in range(GENERATIONS):
#     scored = sorted([(fitness(W), W) for W in population], key=lambda x: x[0], reverse=True)
#     parents = [W for _, W in scored[:5]]
#     new_pop = parents.copy()

#     while len(new_pop) < POP_SIZE:
#         p1, p2 = random.sample(parents, 2)
#         child = 0.5 * (p1 + p2)

#         if random.random() < MUT_RATE:
#             child += DELTA * torch.randn_like(child)

#         new_pop.append(child)

#     population = new_pop
#     print(f"GA Gen {gen+1}, Best Val Loss: {-scored[0][0]:.4f}")

# best_W_high = scored[0][1]

# # ======================
# # STEP 4: PROJECT BACK
# # ======================
# W_final = R @ best_W_high   # (512 × 100)
# W_final = W_sgd #R @ W_high_init   # (512 × 100)

# classifier = nn.Linear(512, 100).to(device)
# classifier.weight.data = W_final.T.to(device)
# classifier.bias.data = b_sgd.to(device)

optimizer = optim.AdamW(
    classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)



print("\n=== FINAL SGD FINETUNE ===")
for epoch in range(10):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Finetune Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")



NUM_CLASSES = 100
lambda_reg = 1e-3

# One-hot labels
Y_tr = torch.zeros(len(y_tr), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr)), y_tr] = 1.0

H = X_tr          # (N × 512)

# Closed-form ridge regression
A = H.T @ H + lambda_reg * torch.eye(H.shape[1])
B = H.T @ Y_tr

W_elm = torch.linalg.solve(A, B)   # (512 × 100)
with torch.no_grad():
    logits = X_test @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY (ELM):", acc.item() * 100)



classifier.eval()
with torch.no_grad():
    logits = classifier(X_test.to(device)) 
    acc = (logits.argmax(1).cpu() == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY:", acc.item() * 100)
'''

In [ ]:
# ======================
# ELM / RANDOM PROJECTION + RIDGE REGRESSION
# ======================
'''NUM_CLASSES = 100
D_HIDDEN = 1000
lambda_reg = 1e-3

# Move to CPU for closed-form (safer)
X_tr_cpu = X_tr.cpu()
X_test_cpu = X_test.cpu()
y_tr_cpu = y_tr
y_test_cpu = y_test

# ======================
# Random Hidden Weights (ELM)
# ======================
Wr = torch.randn(D_HIDDEN, X_tr_cpu.shape[1]) * 0.1
br = torch.randn(D_HIDDEN) * 0.1

# ======================
# Hidden Layer Output
# H = X @ Wᵀ + b
# ======================
H_tr = X_tr_cpu @ Wr.T + br
H_tr = torch.relu(H_tr)

H_test = X_test_cpu @ Wr.T + br
H_test = torch.relu(H_test)

# ======================
# One-hot labels
# ======================
Y_tr = torch.zeros(len(y_tr_cpu), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr_cpu)), y_tr_cpu] = 1.0

# ======================
# Closed-form Ridge Regression
# ======================
A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
B = H_tr.T @ Y_tr

W_elm = torch.linalg.solve(A, B)   # (D_HIDDEN × 100)

# ======================
# Evaluation
# ======================
with torch.no_grad():
    logits = H_test @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test_cpu).float().mean()

print("\n🔥 FINAL TEST ACCURACY (ELM):", acc.item() * 100)
'''

In [ ]:
'''import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet18

# ======================
# Device
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset
# ======================

# ======================
# ORTHOGONAL PROJECTION
# ======================
D_HIGH = 5000   # you can try 1000 / 3000 as well

R = torch.randn(512, D_HIGH)
R, _ = torch.linalg.qr(R)   # 🔥 orthogonal basis

# Project features
X_tr_h  = X_tr @ R
X_val_h = X_val @ R
X_te_h  = X_test @ R

# ======================
# CLASSIFIER IN PROJECTED SPACE
# ======================
classifier = nn.Linear(D_HIGH, 100).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

train_loader = DataLoader(
    TensorDataset(X_tr_h.to(device), y_tr.to(device)),
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_h.to(device), y_val.to(device)),
    batch_size=512
)

# ======================
# TRAINING
# ======================
print("\n=== TRAINING (ORTHOGONAL PROJECTION ONLY) ===")
for epoch in range(20):
    classifier.train()
    train_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validation
    classifier.eval()
    with torch.no_grad():
        val_loss = 0
        for xb, yb in val_loader:
            val_loss += criterion(classifier(xb), yb).item()

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Val Loss: {val_loss/len(val_loader):.4f}"
    )

# ======================
# FINAL EVALUATION
# ======================
classifier.eval()
with torch.no_grad():
    logits = classifier(X_te_h.to(device))
    acc = (logits.argmax(1).cpu() == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY:", acc.item() * 100)
'''

In [ ]:
'''import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet18
import random

classifier = nn.Linear(512, 100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(classifier.parameters(), lr=0.1, momentum=0.9)

loader = DataLoader(
    TensorDataset(X_tr.to(device), y_tr.to(device)),
    batch_size=256, shuffle=True
)

print("\n=== SGD TRAINING ===")
for epoch in range(20):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")

# Save SGD weights
W_sgd = classifier.weight.data.T.cpu()   # (512 × 100)
b_sgd = classifier.bias.data.cpu()

# ======================
# STEP 2: ORTHOGONAL RANDOM PROJECTION
# ======================
D_HIGH = 5000

R = torch.randn(512, D_HIGH)
R, _ = torch.linalg.qr(R)   # 🔥 orthogonal

X_tr_h  = X_tr @ R
X_val_h = X_val @ R
X_te_h  = X_test @ R

W_high_init = R.T @ W_sgd   # (D_HIGH × 100)

# ======================
# STEP 3: GENETIC ALGORITHM (VALIDATION-BASED)
# ======================
POP_SIZE = 20
GENERATIONS = 20
MUT_RATE = 0.3
DELTA = 0.01

def fitness(W):
    with torch.no_grad():
        logits = X_val_h @ W
        loss = criterion(logits, y_val)
    return -loss.item()

population = [
    W_high_init + 0.01 * torch.randn_like(W_high_init)
    for _ in range(POP_SIZE)
]

print("\n=== GA OPTIMIZATION ===")
for gen in range(GENERATIONS):
    scored = sorted([(fitness(W), W) for W in population], key=lambda x: x[0], reverse=True)
    parents = [W for _, W in scored[:5]]
    new_pop = parents.copy()

    while len(new_pop) < POP_SIZE:
        p1, p2 = random.sample(parents, 2)
        child = 0.5 * (p1 + p2)

        if random.random() < MUT_RATE:
            child += DELTA * torch.randn_like(child)

        new_pop.append(child)

    population = new_pop
    print(f"GA Gen {gen+1}, Best Val Loss: {-scored[0][0]:.4f}")

best_W_high = scored[0][1]

# ======================
# STEP 4: PROJECT BACK
# ======================
W_final = R @ best_W_high   # (512 × 100)

# ======================
# STEP 5: FINAL SGD FINE-TUNING (🔥 CRITICAL)
# ======================
classifier = nn.Linear(512, 100).to(device)
classifier.weight.data = W_final.T.to(device)
classifier.bias.data = b_sgd.to(device)

optimizer = optim.AdamW(
    classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


loader = DataLoader(
    TensorDataset(X_tr.to(device), y_tr.to(device)),
    batch_size=256, shuffle=True
)

print("\n=== FINAL SGD FINETUNE ===")
for epoch in range(10):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Finetune Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")

# ======================
# FINAL EVALUATION
# ======================
classifier.eval()
with torch.no_grad():
    logits = classifier(X_test.to(device)) 
    acc = (logits.argmax(1).cpu() == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY:", acc.item() * 100)
'''

In [ ]:
'''def qpso_optimize_fc(model, trainloader, particles=10, iterations=10, beta=0.75):
    model.eval()

    # Extract features
    features, labels = [], []
    with torch.no_grad():
        for x, y in trainloader:
            x = x.to(device)

            # Forward till avgpool
            x = model.conv1(x)
            x = model.bn1(x)
            x = model.relu(x)
            x = model.maxpool(x)
            x = model.layer1(x)
            x = model.layer2(x)
            x = model.layer3(x)
            x = model.layer4(x)
            x = model.avgpool(x)

            x = torch.flatten(x, 1)
            features.append(x.cpu())
            labels.append(y)

    X = torch.cat(features).to(device)
    Y = torch.cat(labels).to(device)

    dim = model.fc.weight.numel()
    gbest = model.fc.weight.data.clone().view(-1).to(device)

   
    particles_pos = [
        gbest + torch.randn(dim, device=device) * 0.1
        for _ in range(particles)
    ]

    mbest = torch.mean(torch.stack(particles_pos), dim=0)

    for it in range(iterations):
        for i in range(particles):
            w = particles_pos[i].view_as(model.fc.weight)
            model.fc.weight.data = w

            outputs = torch.matmul(X, w.t())
            loss = criterion(outputs, Y)

            gbest_loss = criterion(
                torch.matmul(X, gbest.view_as(w).t()), Y
            )

            if loss.item() < gbest_loss.item():
                gbest = w.view(-1).clone()

            u = torch.rand(dim, device=device)
            direction = torch.sign(torch.rand(dim, device=device) - 0.5)
            particles_pos[i] = mbest + beta * direction * torch.log(1 / u) * (gbest - particles_pos[i])

        mbest = torch.mean(torch.stack(particles_pos), dim=0)
        print(f"[QPSO] Iteration {it+1}/{iterations}")

    model.fc.weight.data = gbest.view_as(model.fc.weight)

# ======================
# Evaluation
# ======================
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    print("Test Accuracy:", 100 * correct / total)


print("\nStarting Quantum-Inspired Optimization...")
qpso_optimize_fc(model, trainloader)

evaluate(model)
'''

# New classifier with Ridge

In [ ]:
'''N = int(0.9 * len(X_train))
X_tr, X_val = X_train[:N], X_train[N:]
y_tr, y_val = y_train[:N], y_train[N:]

# ======================
# INITIAL CLASSIFIER (SGD)
# ======================
classifier = nn.Linear(512, 100).to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    classifier.parameters(),
    lr=0.01,
    momentum=0.9
)

train_loader = DataLoader(
    TensorDataset(X_tr.to(device), y_tr.to(device)),
    batch_size=256,
    shuffle=True
)

print("\n=== INITIAL SGD TRAINING ===")
for epoch in range(20):
    loss_sum = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(train_loader):.4f}")



# ======================
# ELM CLASSIFIER (NO SGD)
# ======================
NUM_CLASSES = 100
lambda_reg = 1e-3

# One-hot labels
Y_tr = torch.zeros(len(y_tr), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr)), y_tr] = 1.0

H = X_tr_h    # (N × D_HIGH)

# Closed-form ridge regression
A = H.T @ H + lambda_reg * torch.eye(H.shape[1])
B = H.T @ Y_tr

W_elm = torch.linalg.solve(A, B)   # (D_HIGH × 100)

# ======================
# EVALUATION
# ======================
with torch.no_grad():
    logits = X_te_h @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY (ELM):", acc.item() * 100)'''

# ELM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet18
import random

In [ ]:
'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = DataLoader(testset, batch_size=64, shuffle=False)
'''

In [ ]:
'''model = resnet18(pretrained=True)
model.fc = nn.Identity()
model = model.to(device)
model.eval()

for p in model.parameters():
    p.requires_grad = False


def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats), torch.cat(labels)

print("Extracting features...")
X_train, y_train = extract_features(trainloader)
X_test,  y_test  = extract_features(testloader)


N = int(0.9 * len(X_train))
X_tr, X_val = X_train[:N], X_train[N:]
y_tr, y_val = y_train[:N], y_train[N:]'''

In [ ]:
classifier = nn.Linear(512, 100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(classifier.parameters(), lr=0.01, momentum=0.9)

loader = DataLoader(
    TensorDataset(X_tr.to(device), y_tr.to(device)),
    batch_size=256, shuffle=True
)


In [ ]:
'''
print("\n=== SGD TRAINING ===")
for epoch in range(20):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")

# Save SGD weights
W_sgd = classifier.weight.data.T.cpu()   # (512 × 100)
b_sgd = classifier.bias.data.cpu()

optimizer = optim.AdamW(
    classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)



print("\n=== FINAL SGD FINETUNE ===")
for epoch in range(10):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Finetune Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")



NUM_CLASSES = 100
lambda_reg = 1e-3

# One-hot labels
Y_tr = torch.zeros(len(y_tr), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr)), y_tr] = 1.0

H = X_tr          # (N × 512)

# Closed-form ridge regression
A = H.T @ H + lambda_reg * torch.eye(H.shape[1])
B = H.T @ Y_tr

W_elm = torch.linalg.solve(A, B)   # (512 × 100)
with torch.no_grad():
    logits = X_test @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY (ELM):", acc.item() * 100)



classifier.eval()
with torch.no_grad():
    logits = classifier(X_test.to(device)) 
    acc = (logits.argmax(1).cpu() == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY:", acc.item() * 100)
'''

In [ ]:
NUM_CLASSES = 100
D_HIDDEN = 10000
lambda_reg = 1e-3

# Move to CPU for closed-form (safer)
X_tr_cpu = X_tr.cpu()
X_test_cpu = X_test.cpu()
y_tr_cpu = y_tr
y_test_cpu = y_test

print("X_tr shape     :", X_tr_cpu.shape)    # (N × 512)
print("X_test shape   :", X_test_cpu.shape)

# ======================
# Random Hidden Weights
# ======================
Wr = torch.randn(D_HIDDEN, X_tr_cpu.shape[1]) * 0.1
#br = torch.randn(D_HIDDEN) * 0.1

print("Wr shape       :", Wr.shape)            # (D_HIDDEN × 512)
#print("br shape       :", br.shape)            # (D_HIDDEN,)

# ======================
# Hidden Layer Output
# H = X @ Wᵀ + b
# ======================
H_tr = X_tr_cpu @ Wr.T 
H_tr = torch.relu(H_tr)

H_test = X_test_cpu @ Wr.T 
H_test = torch.relu(H_test)

print("H_tr shape     :", H_tr.shape)          # (N × D_HIDDEN)
print("H_test shape   :", H_test.shape)

# ======================
# One-hot labels
# ======================
Y_tr = torch.zeros(len(y_tr_cpu), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr_cpu)), y_tr_cpu] = 1.0

print("Y_tr shape     :", Y_tr.shape)          # (N × 100)

# ======================
# Closed-form Ridge Regression
# ======================
A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
B = H_tr.T @ Y_tr

print("A shape        :", A.shape)             # (D_HIDDEN × D_HIDDEN)
print("B shape        :", B.shape)             # (D_HIDDEN × 100)

W_elm = torch.linalg.solve(A, B)               # (D_HIDDEN × 100)

print("W_elm shape    :", W_elm.shape)

# ======================
# Evaluation
# ======================
with torch.no_grad():
    logits = H_test @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test_cpu).float().mean()

print("Logits shape   :", logits.shape)        # (N_test × 100)
print("\n🔥 FINAL TEST ACCURACY (ELM):", acc.item() * 100)


# GA + above

In [ ]:

NUM_CLASSES = 100
D_HIDDEN = 1000
lambda_reg = 1e-3

POP_SIZE = 10
GENERATIONS = 10
MUTATION_RATE = 0.1
ELITE = 2

device = "cpu"
torch.manual_seed(42)
random.seed(42)

# ======================
# DATA (ASSUMED READY)
# X_tr : (N × 512)
# X_test : (N_test × 512)
# y_tr : (N,)
# y_test : (N_test,)
# ======================
X_tr_cpu = X_tr.cpu()
X_test_cpu = X_test.cpu()
y_tr_cpu = y_tr
y_test_cpu = y_test

INPUT_DIM = X_tr_cpu.shape[1]

print("Train features :", X_tr_cpu.shape)
print("Test features  :", X_test_cpu.shape)

# ======================
# ONE-HOT LABELS
# ======================
Y_tr = torch.zeros(len(y_tr_cpu), NUM_CLASSES)
Y_tr[torch.arange(len(y_tr_cpu)), y_tr_cpu] = 1.0

# ======================
# FITNESS FUNCTION
# ======================
def elm_fitness(Wr):
    # Hidden layer
    H_tr = torch.relu(X_tr_cpu @ Wr.T)
    H_te = torch.relu(X_test_cpu @ Wr.T)

    # Ridge regression
    A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
    B = H_tr.T @ Y_tr
    W_out = torch.linalg.solve(A, B)

    # Accuracy
    with torch.no_grad():
        logits = H_te @ W_out
        preds = logits.argmax(dim=1)
        acc = (preds == y_test_cpu).float().mean()

    return acc.item()

# ======================
# INITIAL POPULATION
# ======================
population = [
    torch.randn(D_HIDDEN, INPUT_DIM) * 0.1
    for _ in range(POP_SIZE)
]

# ======================
# GENETIC ALGORITHM
# ======================
for gen in range(GENERATIONS):
    fitness_scores = [elm_fitness(W) for W in population]

    ranked = sorted(
        zip(population, fitness_scores),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Gen {gen+1:02d} | Best Acc: {ranked[0][1]*100:.2f}%")

    # Elitism
    new_population = [ranked[i][0] for i in range(ELITE)]

    # Crossover + Mutation
    while len(new_population) < POP_SIZE:
        p1, p2 = random.sample(ranked[:5], 2)
        W1, W2 = p1[0], p2[0]

        alpha = torch.rand(1)
        child = alpha * W1 + (1 - alpha) * W2

        if random.random() < MUTATION_RATE:
            child += 0.05 * torch.randn_like(child)

        new_population.append(child)

    population = new_population

# ======================
# FINAL TRAINING WITH BEST WEIGHTS
# ======================
best_Wr = ranked[0][0]

H_tr = torch.relu(X_tr_cpu @ best_Wr.T)
H_test = torch.relu(X_test_cpu @ best_Wr.T)

A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
B = H_tr.T @ Y_tr
W_elm = torch.linalg.solve(A, B)

with torch.no_grad():
    logits = H_test @ W_elm
    preds = logits.argmax(dim=1)
    acc = (preds == y_test_cpu).float().mean()

print("\n🔥 FINAL TEST ACCURACY (GA-ELM):", acc.item() * 100)


# Frozen CNN + GA-ELM"

In [ ]:

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = DataLoader(testset, batch_size=64, shuffle=False)

# ======================
# Frozen ResNet18 (FC KEPT)
# ======================
model = resnet18(pretrained=True)   # FC NOT REMOVED
model.eval()
model = model.to(device)

for p in model.parameters():
    p.requires_grad = False

# ======================
# Feature Extraction
# ======================
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)          # (N × 1000)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats), torch.cat(labels)

print("Extracting features...")
X_train, y_train = extract_features(trainloader)
X_test,  y_test  = extract_features(testloader)

# ======================
# Train / Validation Split
# ======================
N = int(0.9 * len(X_train))
X_tr, X_val = X_train[:N], X_train[N:]
y_tr, y_val = y_train[:N], y_train[N:]

X_tr   = X_tr.cpu()
X_val  = X_val.cpu()
X_test = X_test.cpu()

y_tr   = y_tr.cpu()
y_val  = y_val.cpu()
y_test = y_test.cpu()

# ======================
# One-hot labels
# ======================
NUM_CLASSES = 100

def one_hot(y):
    Y = torch.zeros(len(y), NUM_CLASSES)
    Y[torch.arange(len(y)), y] = 1.0
    return Y

Y_tr  = one_hot(y_tr)
Y_val = one_hot(y_val)

# ======================
# GA + ELM Settings
# ======================
INPUT_DIM = X_tr.shape[1]   # 1000
D_HIDDEN  = 1000            # keep small for GA
POP_SIZE  = 20
GENS      = 10
MUT_RATE  = 0.1
lambda_reg = 1e-3

# ======================
# Fitness Function
# ======================
def fitness(Wr):
    H_tr  = torch.relu(X_tr @ Wr.T)
    H_val = torch.relu(X_val @ Wr.T)

    A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
    B = H_tr.T @ Y_tr
    W_out = torch.linalg.solve(A, B)

    logits = H_val @ W_out
    preds = logits.argmax(1)
    acc = (preds == y_val).float().mean()

    return acc.item()

# ======================
# Genetic Algorithm
# ======================
print("\n=== GA OPTIMIZATION ===")

population = [
    torch.randn(D_HIDDEN, INPUT_DIM) * 0.1
    for _ in range(POP_SIZE)
]

best_Wr = None
best_acc = 0.0

for gen in range(GENS):
    scores = []

    for Wr in population:
        acc = fitness(Wr)
        scores.append(acc)

        if acc > best_acc:
            best_acc = acc
            best_Wr = Wr.clone()

    print(f"Generation {gen+1}/{GENS} | Best Val Acc: {max(scores)*100:.2f}%")

    # Selection (top 50%)
    idx = torch.argsort(torch.tensor(scores), descending=True)
    parents = [population[i] for i in idx[:POP_SIZE // 2]]

    # Crossover + Mutation
    new_population = parents.copy()

    while len(new_population) < POP_SIZE:
        p1, p2 = random.sample(parents, 2)
        mask = torch.rand_like(p1) < 0.5
        child = torch.where(mask, p1, p2)

        mut_mask = torch.rand_like(child) < MUT_RATE
        child[mut_mask] += 0.05 * torch.randn_like(child[mut_mask])

        new_population.append(child)

    population = new_population

# ======================
# FINAL TEST EVALUATION (FREEZE EVERYTHING)
# ======================
print("\n=== FINAL TEST EVALUATION ===")

H_tr   = torch.relu(X_tr @ best_Wr.T)
H_test = torch.relu(X_test @ best_Wr.T)

A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
B = H_tr.T @ Y_tr
W_out = torch.linalg.solve(A, B)

with torch.no_grad():
    logits = H_test @ W_out
    preds = logits.argmax(1)
    acc = (preds == y_test).float().mean()

print(f"\n🔥 FINAL TEST ACCURACY (Frozen CNN + GA-ELM): {acc.item()*100:.2f}%")


# Frozen CNN + Plain ELM

In [ ]:

NUM_CLASSES = 100

def one_hot(y):
    Y = torch.zeros(len(y), NUM_CLASSES)
    Y[torch.arange(len(y)), y] = 1.0
    return Y

Y_tr = one_hot(y_tr)

# ======================
# Plain ELM Settings (NO GA)
# ======================
INPUT_DIM = X_tr.shape[1]   # 1000
D_HIDDEN  = 1000
lambda_reg = 1e-3

# ======================
# Random Hidden Weights (FIXED)
# ======================
torch.manual_seed(0)   # for fair comparison
Wr = torch.randn(D_HIDDEN, INPUT_DIM) * 0.1

# ======================
# Hidden Layer
# ======================
H_tr   = torch.relu(X_tr @ Wr.T)
H_test = torch.relu(X_test @ Wr.T)

# ======================
# Closed-form Ridge Regression
# ======================
A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
B = H_tr.T @ Y_tr
W_out = torch.linalg.solve(A, B)

# ======================
# Final Test Evaluation
# ======================
with torch.no_grad():
    logits = H_test @ W_out
    preds = logits.argmax(1)
    acc = (preds == y_test).float().mean()

print(f"\n🔥 FINAL TEST ACCURACY (Frozen CNN + Plain ELM): {acc.item()*100:.2f}%")


# GA + Normal Classifier

In [ ]:

# ======================
# GA SETTINGS
# ======================
INPUT_DIM = X_tr.shape[1]   # 1000
D_HIDDEN  = 1000
POP_SIZE  = 20
GENS      = 10
MUT_RATE  = 0.1

# ======================
# Fitness: NORMAL CLASSIFIER (not ELM)
# ======================
def fitness(Wr):
    # Project features
    H_tr  = torch.relu(X_tr @ Wr.T)
    H_val = torch.relu(X_val @ Wr.T)

    clf = nn.Linear(D_HIDDEN, 100)
    optimizer = optim.SGD(clf.parameters(), lr=0.05)
    criterion = nn.CrossEntropyLoss()

    # VERY short training (cheap fitness)
    for _ in range(3):
        optimizer.zero_grad()
        loss = criterion(clf(H_tr), y_tr)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        preds = clf(H_val).argmax(1)
        acc = (preds == y_val).float().mean()

    return acc.item()

# ======================
# Genetic Algorithm
# ======================
print("\n=== GA OPTIMIZATION (Normal Classifier Fitness) ===")

population = [
    torch.randn(D_HIDDEN, INPUT_DIM) * 0.1
    for _ in range(POP_SIZE)
]

best_Wr = None
best_acc = 0.0

for gen in range(GENS):
    scores = []

    for Wr in population:
        acc = fitness(Wr)
        scores.append(acc)

        if acc > best_acc:
            best_acc = acc
            best_Wr = Wr.clone()

    print(f"Generation {gen+1}/{GENS} | Best Val Acc: {max(scores)*100:.2f}%")

    # Selection
    idx = torch.argsort(torch.tensor(scores), descending=True)
    parents = [population[i] for i in idx[:POP_SIZE // 2]]

    # Crossover + Mutation
    new_population = parents.copy()
    while len(new_population) < POP_SIZE:
        p1, p2 = random.sample(parents, 2)
        mask = torch.rand_like(p1) < 0.5
        child = torch.where(mask, p1, p2)

        mut_mask = torch.rand_like(child) < MUT_RATE
        child[mut_mask] += 0.05 * torch.randn_like(child[mut_mask])

        new_population.append(child)

    population = new_population

# ======================
# FINAL TRAINING (NORMAL CLASSIFIER)
# ======================
print("\n=== FINAL TRAINING WITH BEST GA WEIGHTS ===")

H_train = torch.relu(X_train @ best_Wr.T)
H_test  = torch.relu(X_test @ best_Wr.T)

classifier = nn.Linear(D_HIDDEN, 100)
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

loader = DataLoader(
    TensorDataset(H_train, y_train),
    batch_size=256,
    shuffle=True
)

for epoch in range(15):
    loss_sum = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(classifier(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(loader):.4f}")

# ======================
# Final Test Accuracy
# ======================
with torch.no_grad():
    preds = classifier(H_test).argmax(1)
    acc = (preds == y_test).float().mean()

print(f"\n🔥 FINAL TEST ACCURACY (GA + Normal Classifier): {acc.item()*100:.2f}%")


# final GA with finetuning

In [ ]:

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = DataLoader(testset, batch_size=64, shuffle=False)

# ======================
# Frozen ResNet18 (FC KEPT)
# ======================
model = resnet18(pretrained=True)
model.eval()
model = model.to(device)

for p in model.parameters():
    p.requires_grad = False

# ======================
# Feature Extraction
# ======================
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)        # (N × 1000)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats), torch.cat(labels)

print("Extracting features...")
X_all, y_all = extract_features(trainloader)
X_test, y_test = extract_features(testloader)

# ======================
# Train / Validation Split (for GA)
# ======================
N = int(0.9 * len(X_all))
X_tr, X_val = X_all[:N], X_all[N:]
y_tr, y_val = y_all[:N], y_all[N:]

# ======================
# One-hot labels
# ======================
NUM_CLASSES = 100

def one_hot(y):
    Y = torch.zeros(len(y), NUM_CLASSES)
    Y[torch.arange(len(y)), y] = 1.0
    return Y

Y_tr  = one_hot(y_tr)
Y_val = one_hot(y_val)

# ======================
# GA + ELM Settings
# ======================
INPUT_DIM  = X_tr.shape[1]   # 1000
D_HIDDEN   = 1000
POP_SIZE   = 20
GENS       = 10
MUT_RATE   = 0.1
lambda_reg = 1e-3

# ======================
# Fitness Function (GA + ELM)
# ======================
def fitness(Wr):
    # Hidden layer
    H_tr  = torch.relu(X_tr @ Wr.T)
    H_val = torch.relu(X_val @ Wr.T)

    # Closed-form ridge regression
    A = H_tr.T @ H_tr + lambda_reg * torch.eye(D_HIDDEN)
    B = H_tr.T @ Y_tr
    W_out = torch.linalg.solve(A, B)

    # Validation accuracy
    logits = H_val @ W_out
    preds = logits.argmax(1)
    acc = (preds == y_val).float().mean()

    return acc.item()

# ======================
# Genetic Algorithm
# ======================
print("\n=== GA OPTIMIZATION ===")

population = [
    torch.randn(D_HIDDEN, INPUT_DIM) * 0.1
    for _ in range(POP_SIZE)
]

best_Wr = None
best_acc = 0.0

for gen in range(GENS):
    scores = []

    for Wr in population:
        acc = fitness(Wr)
        scores.append(acc)

        if acc > best_acc:
            best_acc = acc
            best_Wr = Wr.clone()

    print(f"Generation {gen+1}/{GENS} | Best Val Acc: {max(scores)*100:.2f}%")

    # Selection (top 50%)
    idx = torch.argsort(torch.tensor(scores), descending=True)
    parents = [population[i] for i in idx[:POP_SIZE // 2]]

    # Crossover + Mutation
    new_population = parents.copy()

    while len(new_population) < POP_SIZE:
        p1, p2 = random.sample(parents, 2)
        mask = torch.rand_like(p1) < 0.5
        child = torch.where(mask, p1, p2)

        mut_mask = torch.rand_like(child) < MUT_RATE
        child[mut_mask] += 0.05 * torch.randn_like(child[mut_mask])

        new_population.append(child)

    population = new_population

# ======================
# FINAL TRAINING (Train + Val) USING BEST GA WEIGHTS
# ======================
print("\n=== FINAL TRAINING WITH BEST GA WEIGHTS ===")

X_full = torch.cat([X_tr, X_val], dim=0)
y_full = torch.cat([y_tr, y_val], dim=0)
Y_full = one_hot(y_full)

H_full = torch.relu(X_full @ best_Wr.T)
H_test = torch.relu(X_test @ best_Wr.T)

A = H_full.T @ H_full + lambda_reg * torch.eye(D_HIDDEN)
B = H_full.T @ Y_full
W_out_final = torch.linalg.solve(A, B)

# ======================
# FINAL TEST EVALUATION
# ======================
with torch.no_grad():
    logits = H_test @ W_out_final
    preds = logits.argmax(1)
    acc = (preds == y_test).float().mean()

print(f"\n🔥 FINAL TEST ACCURACY (GA-ELM, final trained): {acc.item()*100:.2f}%")


# improvements

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset (STRONG AUGMENTATION)
# ======================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)


In [ ]:

model = resnet18(pretrained=True)
model.fc = nn.Identity()
model = model.to(device)

# Freeze everything
for p in model.parameters():
    p.requires_grad = False

# Unfreeze LAST BLOCK ONLY ( key improvement)
for p in model.layer4.parameters():
    p.requires_grad = True

# ======================
# Linear Classifier (TEMPORARY, for finetuning)
# ======================
classifier = nn.Linear(512, 100).to(device)

optimizer = torch.optim.AdamW(
    list(model.layer4.parameters()) + list(classifier.parameters()),
    lr=3e-4, weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

In [ ]:
#for name, module in model.named_modules():
#    print(name)

In [ ]:

print("\n=== FINETUNING BACKBONE + CLASSIFIER ===")
for epoch in range(8):
    model.train()
    classifier.train()
    loss_sum = 0

    for x, y in trainloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        feats = model(x)
        logits = classifier(feats)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(trainloader):.4f}")

# ======================
# Feature Extraction (AFTER FINETUNE)
# ======================
model.eval()

def extract(loader):
    X, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)
            X.append(f.cpu())
            Y.append(y)
    return torch.cat(X), torch.cat(Y)

print("\nExtracting features...")
X_train, y_train = extract(trainloader)
X_test, y_test = extract(testloader)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ======================
# RIDGE REGRESSION (CLOSED FORM)
# ======================
NUM_CLASSES = 100
lambda_reg = 1e-3

Y_onehot = torch.zeros(len(y_train), NUM_CLASSES)
Y_onehot[torch.arange(len(y_train)), y_train] = 1.0

H = X_train          # (N × 512)

A = H.T @ H + lambda_reg * torch.eye(512)
B = H.T @ Y_onehot

W_ridge = torch.linalg.solve(A, B)   # (512 × 100)

# ======================
# Evaluation
# ======================
with torch.no_grad():
    logits = X_test @ W_ridge
    preds = logits.argmax(dim=1)
    acc = (preds == y_test).float().mean()

print("\n🔥 FINAL TEST ACCURACY:", acc.item() * 100)


# Incremental freezing

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)


In [ ]:


# ======================
# Model
# ======================
model = resnet18(pretrained=True)
model.fc = nn.Identity()
model = model.to(device)

classifier = nn.Linear(512, 100).to(device)
criterion = nn.CrossEntropyLoss()

# ======================
# Helper functions
# ======================
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_layer(model, layer_name):
    for p in getattr(model, layer_name).parameters():
        p.requires_grad = True

def compute_accuracy(model, classifier, loader):
    model.eval()
    classifier.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            feats = model(x)
            logits = classifier(feats)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

# ======================
# Progressive Fine-Tuning
# ======================
stages = [
    ("layer1", 5),
    ("layer2", 5),
    ("layer3", 5),
    ("layer4", 10),
    ("fc_only", 10),
]

for stage, epochs in stages:
    print(f"\n🚀 TRAINING STAGE: {stage.upper()}")

    freeze_all(model)

    if stage != "fc_only":
        unfreeze_layer(model, stage)

    for p in classifier.parameters():
        p.requires_grad = True

    trainable_params = list(classifier.parameters())
    if stage != "fc_only":
        trainable_params += list(getattr(model, stage).parameters())

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=3e-4,
        weight_decay=1e-4
    )

    for epoch in range(epochs):
        model.train()
        classifier.train()
        loss_sum = 0.0

        for x, y in trainloader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            feats = model(x)
            logits = classifier(feats)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

        train_acc = compute_accuracy(model, classifier, trainloader)
        test_acc = compute_accuracy(model, classifier, testloader)

        print(
            f"Stage {stage} | "
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss {loss_sum/len(trainloader):.4f} | "
            f"Train Acc {train_acc:.2f}% | "
            f"Test Acc {test_acc:.2f}%"
        )

# ======================
# Feature Extraction
# ======================
model.eval()

def extract_features(loader):
    X, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            feats = model(x)
            X.append(feats.cpu())
            Y.append(y)
    return torch.cat(X), torch.cat(Y)

print("\n📦 Extracting features...")
X_train, y_train = extract_features(trainloader)
X_test, y_test = extract_features(testloader)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ======================
# Ridge Regression (Closed Form)
# ======================
NUM_CLASSES = 100
lambda_reg = 1e-3

Y_onehot = torch.zeros(len(y_train), NUM_CLASSES)
Y_onehot[torch.arange(len(y_train)), y_train] = 1.0

H = X_train
A = H.T @ H + lambda_reg * torch.eye(512)
B = H.T @ Y_onehot

W_ridge = torch.linalg.solve(A, B)

# ======================
# Final Evaluation
# ======================
with torch.no_grad():
    logits = X_test @ W_ridge
    preds = logits.argmax(dim=1)
    acc = (preds == y_test).float().mean()

print("\n🔥 FINAL RIDGE TEST ACCURACY:", acc.item() * 100)


In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset
# ======================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# ======================
# Model (NO FREEZING)
# ======================
model = resnet18(pretrained=True)
model.fc = nn.Linear(512, 100)   # CIFAR-100
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# ======================
# Accuracy function
# ======================
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

# ======================
# Training Loop
# ======================
EPOCHS = 10

print("\n🚀 FULL FINETUNING (NO FREEZING)")
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0.0

    for x, y in trainloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    train_acc = accuracy(model, trainloader)
    test_acc = accuracy(model, testloader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss {loss_sum/len(trainloader):.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )


Using device: cuda


100%|██████████| 169M/169M [00:17<00:00, 9.59MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]



🚀 FULL FINETUNING (NO FREEZING)
Epoch 1/10 | Loss 1.5498 | Train Acc 72.44% | Test Acc 67.58%
Epoch 2/10 | Loss 0.8278 | Train Acc 80.65% | Test Acc 72.89%
Epoch 3/10 | Loss 0.6223 | Train Acc 84.70% | Test Acc 74.93%
Epoch 4/10 | Loss 0.4959 | Train Acc 86.22% | Test Acc 73.93%
Epoch 5/10 | Loss 0.4076 | Train Acc 90.20% | Test Acc 76.33%
Epoch 6/10 | Loss 0.3256 | Train Acc 91.58% | Test Acc 76.77%
Epoch 7/10 | Loss 0.2809 | Train Acc 92.43% | Test Acc 76.41%
Epoch 8/10 | Loss 0.2359 | Train Acc 92.40% | Test Acc 75.74%
Epoch 9/10 | Loss 0.1972 | Train Acc 93.21% | Test Acc 75.87%
Epoch 10/10 | Loss 0.1914 | Train Acc 94.64% | Test Acc 77.14%


# all layers froozen

In [ ]:

# ======================
# Model (FREEZE ALL)
# ======================
model = resnet18(pretrained=True)

# 🔒 Freeze entire ResNet
for param in model.parameters():
    param.requires_grad = False

# Replace final layer (trainable)
model.fc = nn.Linear(512, 100)

model = model.to(device)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()

# 🔥 Train ONLY the FC layer
optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ======================
# Accuracy Function
# ======================
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

# ======================
# Training Loop
# ======================
EPOCHS = 10

print("\n🚀 TRAINING (RESNET FROZEN, FC ONLY)")
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0.0

    for x, y in trainloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    train_acc = accuracy(model, trainloader)
    test_acc = accuracy(model, testloader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss {loss_sum/len(trainloader):.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )


🚀 TRAINING (RESNET FROZEN, FC ONLY)
Epoch 1/10 | Loss 2.7703 | Train Acc 49.58% | Test Acc 47.91%
Epoch 2/10 | Loss 1.8883 | Train Acc 53.77% | Test Acc 51.80%
Epoch 3/10 | Loss 1.7258 | Train Acc 56.21% | Test Acc 53.83%
Epoch 4/10 | Loss 1.6352 | Train Acc 57.40% | Test Acc 55.50%
Epoch 5/10 | Loss 1.5835 | Train Acc 57.49% | Test Acc 55.70%
Epoch 6/10 | Loss 1.5581 | Train Acc 58.73% | Test Acc 56.33%
Epoch 7/10 | Loss 1.5322 | Train Acc 58.89% | Test Acc 56.26%
Epoch 8/10 | Loss 1.5093 | Train Acc 59.64% | Test Acc 57.06%


In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset
# ======================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# ======================
# Model (FREEZE LAYER 1–4)
# ======================
model = resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer
model.fc = nn.Linear(512, 100)

# Unfreeze FC layer only
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),   # only train FC
    lr=3e-4,
    weight_decay=1e-4
)

# ======================
# Accuracy function
# ======================
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

# ======================
# Training Loop
# ======================
EPOCHS = 10

print("\n🚀 FREEZING LAYER 1–4 (Training FC only)")
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0.0

    for x, y in trainloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    train_acc = accuracy(model, trainloader)
    test_acc = accuracy(model, testloader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss {loss_sum/len(trainloader):.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )

Using device: cuda


100%|██████████| 169M/169M [00:04<00:00, 34.2MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 221MB/s]



🚀 FREEZING LAYER 1–4 (Training FC only)
Epoch 1/10 | Loss 3.6251 | Train Acc 40.70% | Test Acc 40.05%
Epoch 2/10 | Loss 2.5291 | Train Acc 48.19% | Test Acc 46.95%
Epoch 3/10 | Loss 2.1401 | Train Acc 51.15% | Test Acc 49.67%
Epoch 4/10 | Loss 1.9489 | Train Acc 53.07% | Test Acc 51.17%
Epoch 5/10 | Loss 1.8396 | Train Acc 54.30% | Test Acc 52.50%
Epoch 6/10 | Loss 1.7666 | Train Acc 55.65% | Test Acc 53.21%
Epoch 7/10 | Loss 1.7109 | Train Acc 55.96% | Test Acc 53.98%
Epoch 8/10 | Loss 1.6696 | Train Acc 56.75% | Test Acc 54.65%
Epoch 9/10 | Loss 1.6393 | Train Acc 57.05% | Test Acc 55.23%
Epoch 10/10 | Loss 1.6106 | Train Acc 57.72% | Test Acc 55.29%


In [1]:
# ===============================
# TPU SETUP (KAGGLE TPU)
# ===============================
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18

# TPU Imports
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

device = xm.xla_device()
print("Using device:", device)

# ===============================
# DATASET
# ===============================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

trainset = torchvision.datasets.CIFAR100(
    "./data", train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR100(
    "./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# ===============================
# MODEL
# ===============================
model = resnet18(pretrained=True)
model.fc = nn.Linear(512, 100)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# ===============================
# ACCURACY FUNCTION
# ===============================
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    para_loader = pl.ParallelLoader(loader, [device])
    loader_device = para_loader.per_device_loader(device)

    with torch.no_grad():
        for x, y in loader_device:
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100 * correct / total


# ===============================
# FREEZING FUNCTION
# ===============================
def freeze_until(layer_number):
    """
    layer_number:
    1 -> freeze layer1
    2 -> freeze layer1+layer2
    3 -> freeze layer1+2+3
    4 -> freeze layer1+2+3+4
    """

    # First unfreeze everything
    for param in model.parameters():
        param.requires_grad = True

    if layer_number >= 1:
        for param in model.layer1.parameters():
            param.requires_grad = False

    if layer_number >= 2:
        for param in model.layer2.parameters():
            param.requires_grad = False

    if layer_number >= 3:
        for param in model.layer3.parameters():
            param.requires_grad = False

    if layer_number >= 4:
        for param in model.layer4.parameters():
            param.requires_grad = False


# ===============================
# TRAINING FUNCTION
# ===============================
def train_epochs(epochs, freeze_level):

    freeze_until(freeze_level)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=3e-4,
        weight_decay=1e-4
    )

    for epoch in range(epochs):

        model.train()
        loss_sum = 0.0

        para_loader = pl.ParallelLoader(trainloader, [device])
        loader_device = para_loader.per_device_loader(device)

        for x, y in loader_device:

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()

            xm.optimizer_step(optimizer)
            xm.mark_step()

            loss_sum += loss.item()

        train_acc = accuracy(model, trainloader)
        test_acc = accuracy(model, testloader)

        print(
            f"[Freeze Level {freeze_level}] "
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss {loss_sum/len(trainloader):.4f} | "
            f"Train {train_acc:.2f}% | "
            f"Test {test_acc:.2f}%"
        )


# ===============================
# INCREMENTAL FREEZING
# ===============================

print("\n🔥 Progressive Freezing Training")

# 0 = Full finetune
train_epochs(5, freeze_level=0)

# Freeze layer1
train_epochs(5, freeze_level=1)

# Freeze layer1 + layer2
train_epochs(5, freeze_level=2)

# Freeze layer1 + layer2 + layer3
train_epochs(5, freeze_level=3)

# Freeze layer1 + layer2 + layer3 + layer4
train_epochs(5, freeze_level=4)

print("✅ Training Complete")

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/tmp/ipykernel_12/2588979613.py:16: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
E0000 00:00:1770957599.119221      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Using device: xla:0


100%|██████████| 169M/169M [00:03<00:00, 52.9MB/s] 
/usr/local/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/usr/local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s] 



🔥 Progressive Freezing Training


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/tmp/ipykernel_12/2588979613.py:145: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


[Freeze Level 0] Epoch 1/5 | Loss 1.5466 | Train 74.47% | Test 69.67%
[Freeze Level 0] Epoch 2/5 | Loss 0.8254 | Train 80.32% | Test 72.94%
[Freeze Level 0] Epoch 3/5 | Loss 0.6279 | Train 84.85% | Test 74.84%
[Freeze Level 0] Epoch 4/5 | Loss 0.5013 | Train 87.52% | Test 75.70%
[Freeze Level 0] Epoch 5/5 | Loss 0.4036 | Train 89.11% | Test 75.68%
[Freeze Level 1] Epoch 1/5 | Loss 0.3589 | Train 90.94% | Test 76.58%
[Freeze Level 1] Epoch 2/5 | Loss 0.2828 | Train 93.00% | Test 77.61%
[Freeze Level 1] Epoch 3/5 | Loss 0.2372 | Train 93.78% | Test 76.90%
[Freeze Level 1] Epoch 4/5 | Loss 0.2014 | Train 94.14% | Test 76.15%
[Freeze Level 1] Epoch 5/5 | Loss 0.1718 | Train 95.55% | Test 77.11%
[Freeze Level 2] Epoch 1/5 | Loss 0.1586 | Train 96.46% | Test 77.92%
[Freeze Level 2] Epoch 2/5 | Loss 0.1340 | Train 96.04% | Test 77.09%
[Freeze Level 2] Epoch 3/5 | Loss 0.1155 | Train 97.35% | Test 78.14%
[Freeze Level 2] Epoch 4/5 | Loss 0.1013 | Train 96.63% | Test 77.99%
[Freeze Level 2] Epo

KeyboardInterrupt: 

In [1]:
pip install torch_xla -f https://storage.googleapis.com/libtpu-releases/index.html

Looking in links: https://storage.googleapis.com/libtpu-releases/index.html

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

# ======================
# TPU Device
# ======================
device = xm.xla_device()
print("Using device:", device)

# ======================
# ImageNet Normalization
# ======================
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

# ======================
# Transforms (Resize to 224)
# ======================
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ======================
# Dataset
# ======================
trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=4)

# ======================
# Model
# ======================
model = resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(512, 100)
model = model.to(device)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)

# ======================
# Evaluation
# ======================
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

# ======================
# Training Loop (TPU)
# ======================
EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    para_loader = pl.ParallelLoader(trainloader, [device])
    train_device_loader = para_loader.per_device_loader(device)

    for x, y in train_device_loader:
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()

        xm.optimizer_step(optimizer)  # IMPORTANT for TPU

        running_loss += loss.item()

    scheduler.step()

    train_acc = evaluate(trainloader)
    test_acc = evaluate(testloader)

    xm.master_print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {running_loss/len(trainloader):.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Test Acc: {test_acc:.2f}%"
    )

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/tmp/ipykernel_12/1172467704.py:15: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
E0000 00:00:1771215460.235522      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Using device: xla:0


100%|██████████| 169M/169M [00:01<00:00, 101MB/s]  
/usr/local/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 212MB/s]
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Epoch [1/100] Loss: 2.8997 Train Acc: 52.34% Test Acc: 63.95%


In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

# ======================
# GPU Device
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# ImageNet Normalization
# ======================
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

# ======================
# Transforms (Resize to 224)
# ======================
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ======================
# Dataset
# ======================
trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=train_transform
)

testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=4)

# ======================
# Model
# ======================
model = resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(512, 100)
model = model.to(device)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)

# ======================
# Evaluation
# ======================
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

# ======================
# Training Loop (GPU)
# ======================
EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    for x, y in trainloader:

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()

        optimizer.step()   # Normal GPU step

        running_loss += loss.item()

    scheduler.step()

    train_acc = evaluate(trainloader)
    test_acc = evaluate(testloader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {running_loss/len(trainloader):.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Test Acc: {test_acc:.2f}%"
    )

Using device: cuda


100%|██████████| 169M/169M [00:03<00:00, 48.5MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 221MB/s]


Epoch [1/100] Loss: 2.8868 Train Acc: 53.36% Test Acc: 63.95%
Epoch [2/100] Loss: 2.2237 Train Acc: 58.41% Test Acc: 69.17%
Epoch [3/100] Loss: 2.0880 Train Acc: 62.64% Test Acc: 71.42%
Epoch [4/100] Loss: 2.0060 Train Acc: 65.14% Test Acc: 74.43%
Epoch [5/100] Loss: 1.9461 Train Acc: 67.56% Test Acc: 75.14%
Epoch [6/100] Loss: 1.9035 Train Acc: 68.13% Test Acc: 75.82%
Epoch [7/100] Loss: 1.8587 Train Acc: 70.30% Test Acc: 76.54%
Epoch [8/100] Loss: 1.8275 Train Acc: 71.36% Test Acc: 77.23%
Epoch [9/100] Loss: 1.7946 Train Acc: 71.06% Test Acc: 76.46%
Epoch [10/100] Loss: 1.7709 Train Acc: 72.03% Test Acc: 76.53%
Epoch [11/100] Loss: 1.7495 Train Acc: 72.96% Test Acc: 76.71%
Epoch [12/100] Loss: 1.7310 Train Acc: 73.75% Test Acc: 76.51%
Epoch [13/100] Loss: 1.7093 Train Acc: 73.89% Test Acc: 76.52%
Epoch [14/100] Loss: 1.6825 Train Acc: 75.83% Test Acc: 77.82%
Epoch [15/100] Loss: 1.6750 Train Acc: 75.15% Test Acc: 76.26%
Epoch [16/100] Loss: 1.6526 Train Acc: 76.00% Test Acc: 78.02%
E

KeyboardInterrupt: 

# Random freezing

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import random
import copy


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ---------------- DATA ----------------
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)


Using: cuda


100%|██████████| 169M/169M [00:15<00:00, 11.1MB/s] 


In [3]:
# ---------------- MODEL ----------------
model = resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 100)
model = model.to(device)

# ---------------- FREEZE ALL ----------------
for p in model.parameters():
    p.requires_grad = False

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 236MB/s]


In [5]:
# ---------------- EVALUATE ----------------
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in testloader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            _,pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return correct/total


In [10]:
# ---------------- RANDOMIZED TRAINING ----------------
best_acc = evaluate(model)
print("Initial acc:", best_acc)

# store layer names + modules
layers = {
    "layer1": model.layer1,
    "layer2": model.layer2,
    "layer3": model.layer3,
    "layer4": model.layer4
}

for i in range(50):

    # choose random layer
    layer_name, chosen = random.choice(list(layers.items()))
    print("\nIteration:", i+1)
    print("Unfreezing:", layer_name)

    # save weights
    old_weights = copy.deepcopy(model.state_dict())

    # unfreeze chosen
    for p in chosen.parameters():
        p.requires_grad = True

    optimizer = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=0.001)
    loss_fn = torch.nn.CrossEntropyLoss()

    # train few epochs
    model.train()
    for epoch in range(2):
        for x,y in trainloader:
            x,y=x.to(device),y.to(device)

            optimizer.zero_grad()
            out=model(x)
            loss=loss_fn(out,y)
            loss.backward()
            optimizer.step()

    acc1 = evaluate(model)
    print("New acc:", acc1)

    # decision
    if acc1 > best_acc:
        best_acc = acc1
        print("✅ Improved — keeping weights")
    else:
        model.load_state_dict(old_weights)
        print("❌ No improvement — reverted")

    # freeze again
    for p in model.parameters():
        p.requires_grad = False

print("\nFinal Best Accuracy:", best_acc)

Initial acc: 0.0053

Iteration: 1
Unfreezing: layer4
New acc: 0.681
✅ Improved — keeping weights

Iteration: 2
Unfreezing: layer4
New acc: 0.6849
✅ Improved — keeping weights

Iteration: 3
Unfreezing: layer1
New acc: 0.7035
✅ Improved — keeping weights

Iteration: 4
Unfreezing: layer3
New acc: 0.7084
✅ Improved — keeping weights

Iteration: 5
Unfreezing: layer4
New acc: 0.7074
❌ No improvement — reverted

Iteration: 6
Unfreezing: layer2
New acc: 0.7154
✅ Improved — keeping weights

Iteration: 7
Unfreezing: layer1
New acc: 0.7201
✅ Improved — keeping weights

Iteration: 8
Unfreezing: layer4
New acc: 0.7058
❌ No improvement — reverted

Iteration: 9
Unfreezing: layer1
New acc: 0.7263
✅ Improved — keeping weights

Iteration: 10
Unfreezing: layer3
New acc: 0.7277
✅ Improved — keeping weights

Iteration: 11
Unfreezing: layer1


KeyboardInterrupt: 

# inlcuding overiftting

In [6]:
def accuracy_loader(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            _,pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return correct/total

In [ ]:
# ---------------- RANDOMIZED TRAINING ----------------
best_acc = accuracy_loader(model, testloader)
print("Initial Test acc:", best_acc)

layers = {
    "layer1": model.layer1,
    "layer2": model.layer2,
    "layer3": model.layer3,
    "layer4": model.layer4
}

for i in range(50):

    layer_name, chosen = random.choice(list(layers.items()))
    print("\nIteration:", i+1)
    print("Unfreezing:", layer_name)

    old_weights = copy.deepcopy(model.state_dict())

    for p in chosen.parameters():
        p.requires_grad = True

    optimizer = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=0.001)
    loss_fn = torch.nn.CrossEntropyLoss()

    # ---- training ----
    model.train()
    for epoch in range(2):
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out,y)
            loss.backward()
            optimizer.step()

    # ---- accuracies ----
    train_acc = accuracy_loader(model, trainloader)
    test_acc = accuracy_loader(model, testloader)

    print("Train acc:", train_acc)
    print("Test acc :", test_acc)

    # overfitting indicator
    if train_acc - test_acc > 0.15:
        print("⚠️ Possible Overfitting")

    # decision
    if test_acc > best_acc:
        best_acc = test_acc
        print("✅ Improved — keeping weights")
    else:
        model.load_state_dict(old_weights)
        print("❌ No improvement — reverted")

    # freeze again
    for p in model.parameters():
        p.requires_grad = False

print("\nFinal Best Test Accuracy:", best_acc)

Initial Test acc: 0.0109

Iteration: 1
Unfreezing: layer2
Train acc: 0.14608
Test acc : 0.1353
✅ Improved — keeping weights

Iteration: 2
Unfreezing: layer2
Train acc: 0.21972
Test acc : 0.1967
✅ Improved — keeping weights

Iteration: 3
Unfreezing: layer1
Train acc: 0.21956
Test acc : 0.2004
✅ Improved — keeping weights

Iteration: 4
Unfreezing: layer1
Train acc: 0.22872
Test acc : 0.2076
✅ Improved — keeping weights

Iteration: 5
Unfreezing: layer1
Train acc: 0.23672
Test acc : 0.2109
✅ Improved — keeping weights

Iteration: 6
Unfreezing: layer4
Train acc: 0.62284
Test acc : 0.5141
✅ Improved — keeping weights

Iteration: 7
Unfreezing: layer1
Train acc: 0.65266
Test acc : 0.5448
✅ Improved — keeping weights

Iteration: 8
Unfreezing: layer3
Train acc: 0.70334
Test acc : 0.588
✅ Improved — keeping weights

Iteration: 9
Unfreezing: layer1
Train acc: 0.7076
Test acc : 0.5933
✅ Improved — keeping weights

Iteration: 10
Unfreezing: layer4
Train acc: 0.83304
Test acc : 0.6133
⚠️ Possible Ove

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import random
import copy

# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ---------------- DATA (augmentation reduces overfitting) ----------------
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR100('./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR100('./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# ---------------- MODEL (pretrained helps a lot) ----------------
model = resnet18(weights="IMAGENET1K_V1")

# dropout added to reduce overfitting
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(512,100)
)

model = model.to(device)

# ---------------- FREEZE ALL ----------------
for p in model.parameters():
    p.requires_grad = False

# accuracy function
def accuracy_loader(model, loader):
    model.eval()
    correct,total=0,0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            out=model(x)
            _,pred=out.max(1)
            total+=y.size(0)
            correct+=pred.eq(y).sum().item()
    return correct/total

best_acc = accuracy_loader(model,testloader)
print("Initial test acc:",best_acc)

layers = {
    "layer1": model.layer1,
    "layer2": model.layer2,
    "layer3": model.layer3,
    "layer4": model.layer4
}

# ---------------- RANDOMIZED TRAINING ----------------
for i in range(40):

    print("\nIteration",i+1)

    # choose 1 or 2 layers randomly
    chosen_names = random.sample(list(layers.keys()), random.choice([1,2]))
    print("Unfreezing:",chosen_names)

    old_weights = copy.deepcopy(model.state_dict())

    # unfreeze chosen layers
    for name in chosen_names:
        for p in layers[name].parameters():
            p.requires_grad=True

    # FC ALWAYS train
    for p in model.fc.parameters():
        p.requires_grad=True

    optimizer = torch.optim.Adam(
        filter(lambda p:p.requires_grad,model.parameters()),
        lr=3e-4,
        weight_decay=1e-4   # key for overfitting
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

    loss_fn=torch.nn.CrossEntropyLoss()

    # training
    model.train()
    for epoch in range(3):
        for x,y in trainloader:
            x,y=x.to(device),y.to(device)

            optimizer.zero_grad()
            out=model(x)
            loss=loss_fn(out,y)
            loss.backward()
            optimizer.step()

        scheduler.step()

    train_acc = accuracy_loader(model,trainloader)
    test_acc  = accuracy_loader(model,testloader)

    print("Train:",train_acc)
    print("Test :",test_acc)

    if test_acc > best_acc:
        best_acc=test_acc
        print("✅ Improved")
    else:
        model.load_state_dict(old_weights)
        print("❌ Reverted")

    # freeze again
    for p in model.parameters():
        p.requires_grad=False

print("\nFinal best:",best_acc)

Using: cuda


100%|██████████| 169M/169M [00:02<00:00, 74.9MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


Initial test acc: 0.0122

Iteration 1
Unfreezing: ['layer4']
Train: 0.86996
Test : 0.7553
✅ Improved

Iteration 2
Unfreezing: ['layer3']
Train: 0.91988
Test : 0.7841
✅ Improved

Iteration 3
Unfreezing: ['layer1', 'layer4']
Train: 0.9652
Test : 0.7981
✅ Improved

Iteration 4
Unfreezing: ['layer3']
Train: 0.98046
Test : 0.8005
✅ Improved

Iteration 5
Unfreezing: ['layer3']
Train: 0.9859
Test : 0.7998
❌ Reverted

Iteration 6
Unfreezing: ['layer2', 'layer3']
Train: 0.98422
Test : 0.8029
✅ Improved

Iteration 7
Unfreezing: ['layer4', 'layer2']
Train: 0.98994
Test : 0.8026
❌ Reverted

Iteration 8
Unfreezing: ['layer1', 'layer2']
Train: 0.98678
Test : 0.8025
❌ Reverted

Iteration 9
Unfreezing: ['layer4', 'layer2']
Train: 0.99092
Test : 0.8014
❌ Reverted

Iteration 10
Unfreezing: ['layer1', 'layer2']
Train: 0.98764
Test : 0.8058
✅ Improved

Iteration 11
Unfreezing: ['layer3', 'layer4']
Train: 0.98798
Test : 0.7979
❌ Reverted

Iteration 12
Unfreezing: ['layer3']
Train: 0.98958
Test : 0.7978
❌ R

KeyboardInterrupt: 

# research approach

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import copy
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# Cutout
# =========================
class Cutout(object):
    def __init__(self, size=16):
        self.size = size

    def __call__(self, img):
        h, w = img.shape[1], img.shape[2]
        y = random.randint(0, h)
        x = random.randint(0, w)

        y1 = max(0, y - self.size // 2)
        y2 = min(h, y + self.size // 2)
        x1 = max(0, x - self.size // 2)
        x2 = min(w, x + self.size // 2)

        img[:, y1:y2, x1:x2] = 0
        return img

# =========================
# Data
# =========================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    Cutout(16),
    transforms.Normalize((0.5071,0.4867,0.4408),
                         (0.2675,0.2565,0.2761))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071,0.4867,0.4408),
                         (0.2675,0.2565,0.2761))
])

trainset = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=train_transform)
testset = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=test_transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

# =========================
# Model
# =========================
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

# =========================
# RWP params
# =========================
alpha = 0.5
gamma = 0.01
epochs = 200

def get_perturbed_model(model, gamma):
    perturbed = copy.deepcopy(model)
    with torch.no_grad():
        for p, q in zip(model.parameters(), perturbed.parameters()):
            if p.requires_grad:
                noise = torch.randn_like(p) * gamma * p.norm()
                q.add_(noise)
    return perturbed

# =========================
# Training
# =========================
for epoch in range(epochs):

    model.train()
    train_correct = 0
    train_total = 0
    train_loss_total = 0

    for inputs, targets in trainloader:

        inputs, targets = inputs.to(device), targets.to(device)

        # original
        out1 = model(inputs)
        loss1 = criterion(out1, targets)

        # perturbed
        perturbed = get_perturbed_model(model, gamma)
        out2 = perturbed(inputs)
        loss2 = criterion(out2, targets)

        loss = alpha*loss1 + (1-alpha)*loss2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item()

        _, pred = out1.max(1)
        train_total += targets.size(0)
        train_correct += pred.eq(targets).sum().item()

    scheduler.step()

    train_acc = 100*train_correct/train_total
    train_loss = train_loss_total/len(trainloader)

    # =========================
    # Test
    # =========================
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss_total = 0

    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss_total += loss.item()

            _, pred = outputs.max(1)
            test_total += targets.size(0)
            test_correct += pred.eq(targets).sum().item()

    test_acc = 100*test_correct/test_total
    test_loss = test_loss_total/len(testloader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss {train_loss:.3f} | Train Acc {train_acc:.2f}")
    print(f"Test  Loss {test_loss:.3f} | Test  Acc {test_acc:.2f}")
    print("-"*40)

100%|██████████| 169M/169M [00:08<00:00, 20.4MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


Epoch 1
Train Loss 4.989 | Train Acc 17.10
Test  Loss 2.694 | Test  Acc 30.68
----------------------------------------
Epoch 2
Train Loss 4.609 | Train Acc 31.01
Test  Loss 2.314 | Test  Acc 39.21
----------------------------------------
Epoch 3
Train Loss 4.470 | Train Acc 37.09
Test  Loss 2.087 | Test  Acc 43.84
----------------------------------------
Epoch 4
Train Loss 4.381 | Train Acc 40.76
Test  Loss 1.965 | Test  Acc 46.32
----------------------------------------
Epoch 5
Train Loss 4.322 | Train Acc 43.63
Test  Loss 1.849 | Test  Acc 49.38
----------------------------------------
Epoch 6
Train Loss 4.281 | Train Acc 45.78
Test  Loss 1.833 | Test  Acc 50.20
----------------------------------------
Epoch 7
Train Loss 4.218 | Train Acc 47.63
Test  Loss 1.744 | Test  Acc 52.28
----------------------------------------
Epoch 8
Train Loss 4.184 | Train Acc 49.45
Test  Loss 1.733 | Test  Acc 53.06
----------------------------------------
Epoch 9
Train Loss 4.147 | Train Acc 50.45
Test 